# El costo oculto: ingresos, brecha de genero y penalizacion por informalidad laboral en Chile
### Proyecto de modelado econometrico y machine learning explicable (ESI 2025)
**Autor:** Raul Frugone Zaror  
**Fecha:** Octubre 2026  

---

## 1. Resumen ejecutivo y contexto del problema
Este estudio analiza la estructura salarial en Chile utilizando los microdatos oficiales de la **Encuesta Suplementaria de Ingresos (ESI 2025)** del INE.
Se aplican dos enfoques analiticos complementarios:
1. **Enfoque econometrico de Mincer (WLS con ponderadores muestrales):** Cuantifica el efecto neto causal ceteris paribus de la brecha de genero y la penalizacion salarial por informalidad.
2. **Enfoque de Machine Learning (LightGBM con optimizacion Optuna):** Modela no-linealidades complejas, interacciones entre experiencia y educacion, e identifica los factores determinantes del ingreso mediante importancia de caracteristicas.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar microdatos procesados de la ESI 2025
df = pd.read_csv('microdatos_esi_ocupados_preparados.csv')
print(f'Muestra ocupada analizada: {len(df):,} personas')
print(f'Poblacion nacional expandida: {df["fact_cal_esi"].sum():,.0f} trabajadores')
df[['edad', 'horas_semanales', 'ing_t_p', 'fact_cal_esi']].describe().round(1)

## 2. Estimacion econometrica minceriana (WLS con pesos muestrales)
Se ajusta una ecuacion salarial de corte transversal considerando errores estandar consistentes ante heterocedasticidad (HC1):
$$\ln(w_i) = \beta_0 + \beta_1 \text{es\_mujer}_i + \beta_2 \text{es\_informal}_i + \beta_3 \text{edad}_i + \beta_4 \text{edad}^2_i + \beta_5 \text{horas}_i + \sum_k \delta_k \text{educ}_{ik} + \sum_j \gamma_j \text{region}_{ij} + \varepsilon_i$$

In [ ]:
vars_x = ['es_mujer', 'es_informal', 'edad', 'edad_cuadrado', 'horas_semanales', 'cine11_1d', 'region']
X_wls = pd.get_dummies(df[vars_x], columns=['cine11_1d', 'region'], drop_first=True, dtype=float)
X_wls = sm.add_constant(X_wls)
y = df['log_ingreso']
weights = df['fact_cal_esi']

modelo_wls = sm.WLS(y, X_wls, weights=weights).fit(cov_type='HC1')
print(f'R2 Ponderado del modelo Minceriano: {modelo_wls.rsquared:.4f}')

coef_mujer = modelo_wls.params['es_mujer']
coef_informal = modelo_wls.params['es_informal']
print(f'Efecto neto penalizacion genero (Mujer vs Hombre): {(np.exp(coef_mujer) - 1)*100:.2f}% (p-val: {modelo_wls.pvalues["es_mujer"]:.2e})')
print(f'Efecto neto penalizacion por empleo informal: {(np.exp(coef_informal) - 1)*100:.2f}% (p-val: {modelo_wls.pvalues["es_informal"]:.2e})')

## 3. Modelo de Machine Learning: LightGBM con pesos muestrales
Entrenamos un modelo de gradient boosting con `sample_weight` para capturar patrones no lineales y de interaccion sectorial.

In [ ]:
features_ml = ['es_mujer', 'es_informal', 'edad', 'horas_semanales', 'cine11_1d', 'categoria_ocupacion', 'region', 'r_p_rev4cl_caenes']
X_ml = df[features_ml].copy()
for c in ['cine11_1d', 'categoria_ocupacion', 'region', 'r_p_rev4cl_caenes']:
    X_ml[c] = X_ml[c].astype('category')

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_ml, y, weights, test_size=0.2, random_state=42, stratify=df[['es_mujer', 'es_informal']]
)

lgb_model = lgb.LGBMRegressor(
    n_estimators=180, learning_rate=0.06, max_depth=6, num_leaves=35, random_state=42, verbose=-1
)
lgb_model.fit(X_train, y_train, sample_weight=w_train)

y_pred = lgb_model.predict(X_test)
w_mean = np.average(y_test, weights=w_test)
r2_test = 1.0 - (np.sum(w_test * (y_test - y_pred)**2) / np.sum(w_test * (y_test - w_mean)**2))
rmse_test = np.sqrt(np.average((y_test - y_pred)**2, weights=w_test))
print(f'R2 Ponderado en conjunto Test: {r2_test:.4f}')
print(f'RMSE Ponderado en conjunto Test: {rmse_test:.4f}')

## 4. Importancia de caracteristicas y explicabilidad
Evaluacion de la ganancia relativa de cada variable en la reduccion de error del modelo.

In [ ]:
imp = pd.Series(lgb_model.booster_.feature_importance(importance_type='gain'), index=features_ml)
imp_pct = (imp / imp.sum()) * 100
imp_pct.sort_values(ascending=True).plot(kind='barh', figsize=(8, 4), color='#1f77b4')
plt.title('Importancia de caracteristicas en la explicacion del ingreso laboral (% ganancia)')
plt.xlabel('Porcentaje de ganancia')
plt.tight_layout()
plt.show()